# French Motor Third-Party Liability: Data Intake and Integrity

This notebook assesses the two **freMTPL2** source files before exploratory analysis or modeling. The main checks cover schema, completeness, policy identifiers, exposure and outcome values, and consistency between policy claim counts and claim-level records.

The longer-term project estimates expected claim cost per unit of exposure from driver, vehicle, and geographic characteristics. This notebook establishes whether the raw input files can support that analysis; it does not clean or join the files.

**Provenance:** The local CSVs were exported from the `freMTPL2freq` and `freMTPL2sev` objects in the [CASdatasets R package](https://dutangc.github.io/CASdatasets/reference/freMTPL.html) using `write.csv(..., row.names = FALSE)`. This notebook analyzes those exports.

## Data structure

| File | Unit of Observation | Key fields |
|---|---|---|
| `freMTPL2freq.csv` | Policy exposure period | `IDpol`, `ClaimNb`, `Exposure`, driver, vehicle, and geographic attributes |
| `freMTPL2sev.csv` | Claim amount record | `IDpol`, `ClaimAmount` |

`Exposure` is measured in years. `ClaimNb` counts claims during that exposure period. Multiple severity records may refer to the same policy. `BonusMalus` is below 100 for a bonus and above 100 for a malus; `Area` ranges from A (rural) to F (urban centre). These definitions follow the CASdatasets page. The severity table must be assessed at claim level before combining it with the policy table.

## 1. Load the source files

Read `IDpol` as text so its stored values are preserved during the key and linkage checks.

In [1]:
from pathlib import Path

import pandas as pd

frequency_path = Path("../data/raw/freMTPL2freq.csv")
severity_path = Path("../data/raw/freMTPL2sev.csv")

frequency = pd.read_csv(frequency_path, dtype={"IDpol": "string"})
severity = pd.read_csv(severity_path, dtype={"IDpol": "string"})

print(f"Frequency: {frequency.shape[0]:,} rows, {frequency.shape[1]} columns")
print(f"Severity:  {severity.shape[0]:,} rows, {severity.shape[1]} columns")
display(frequency.head(3), severity.head(3))

Frequency: 677,991 rows, 12 columns
Severity:  26,444 rows, 2 columns


,IDpol,ClaimNb,Exposure,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Area,Density,Region
0,1,0,0.10,5,0,55,50,B12,Regular,D,1217,Rhone-Alpes
1,3,0,0.77,5,0,55,50,B12,Regular,D,1217,Rhone-Alpes
2,5,0,0.75,6,2,52,50,B12,Diesel,B,54,Picardie


,IDpol,ClaimAmount
0,1552,995.20
1,1010996,1128.12
2,4024277,1851.11


## 2. Schema and completeness

Review the observed fields, pandas data types, and missing-value counts. A missing-value count of zero does not by itself establish that a field is valid; range and relationship checks follow.

In [2]:
for name, table in [("Frequency", frequency), ("Severity", severity)]:
    print(f"{name} schema and missing values")
    display(pd.DataFrame({
        "dtype": table.dtypes.astype(str),
        "missing": table.isna().sum(),
        "missing_pct": table.isna().mean().mul(100).round(2),
    }))

Frequency schema and missing values


,dtype,missing,missing_pct
IDpol,string,0,0.0
ClaimNb,int64,0,0.0
Exposure,float64,0,0.0
VehPower,int64,0,0.0
VehAge,int64,0,0.0
DrivAge,int64,0,0.0
BonusMalus,int64,0,0.0
VehBrand,str,0,0.0
VehGas,str,0,0.0
Area,str,0,0.0


Severity schema and missing values


,dtype,missing,missing_pct
IDpol,string,0,0.0
ClaimAmount,float64,0,0.0


## 3. Policy ID representation and linkage

Check the stored ID format, uniqueness in the policy table, and claim-to-policy coverage. Claim IDs may repeat because a policy can have multiple claims.

In [3]:
for name, path, table in [
    ("Frequency", frequency_path, frequency),
    ("Severity", severity_path, severity),
]:
    inferred_type = pd.read_csv(path, usecols=["IDpol"])["IDpol"].dtype
    ids = table["IDpol"]
    print(f"{name} ID audit")
    print("Pandas inferred type:", inferred_type)
    print("Preserved type:", ids.dtype)
    print("Scientific-notation IDs:", ids.str.contains(r"[eE][+-]?\d+", na=False).sum())
    print("IDs containing characters other than digits:", (~ids.str.fullmatch(r"\d+", na=False)).sum())
    print("Distinct IDs:", ids.nunique(dropna=False))
    print("Repeated-ID rows:", ids.duplicated().sum())
    print()

Frequency ID audit
Pandas inferred type: float64
Preserved type: string
Scientific-notation IDs: 1
IDs containing characters other than digits: 1
Distinct IDs: 677991
Repeated-ID rows: 0

Severity ID audit
Pandas inferred type: int64
Preserved type: string
Scientific-notation IDs: 0
IDs containing characters other than digits: 0
Distinct IDs: 24944
Repeated-ID rows: 1500



In [4]:
frequency_ids = set(frequency["IDpol"].dropna())
severity_ids = set(severity["IDpol"].dropna())
unmatched_severity_ids = severity_ids - frequency_ids
unmatched_severity_rows = severity.loc[~severity["IDpol"].isin(frequency_ids)]

print("Severity IDs absent from frequency:", len(unmatched_severity_ids))
print("Severity rows without a frequency policy:", len(unmatched_severity_rows))
print("Example unmatched IDs:", sorted(unmatched_severity_ids)[:10])
print("Example repeated frequency IDs:")
display(frequency.loc[
    frequency["IDpol"].duplicated(keep=False),
    ["IDpol", "ClaimNb", "Exposure"],
].head(10))

Severity IDs absent from frequency: 0
Severity rows without a frequency policy: 0
Example unmatched IDs: []
Example repeated frequency IDs:


,IDpol,ClaimNb,Exposure


## 4. Exposure and outcome checks

Summarize policy exposure, reported claim counts, and claim amounts. Nonpositive exposure, negative claim counts, and nonpositive claim amounts are flagged for review. Exposure above one year is reported separately because its interpretation depends on the portfolio's observation period.

In [5]:
display(frequency[["Exposure", "ClaimNb"]].describe())
display(severity[["ClaimAmount"]].describe())

print("Exposure <= 0:", (frequency["Exposure"] <= 0).sum())
print("Exposure > 1:", (frequency["Exposure"] > 1).sum())
print("ClaimNb < 0:", (frequency["ClaimNb"] < 0).sum())
print("ClaimAmount <= 0:", (severity["ClaimAmount"] <= 0).sum())

,Exposure,ClaimNb
count,677991.000000,677991.000000
mean,0.528743,0.039003
std,0.364441,0.207169
min,0.002732,0.000000
25%,0.180000,0.000000
50%,0.490000,0.000000
75%,0.990000,0.000000
max,2.010000,16.000000


,ClaimAmount
count,2.644400e+04
mean,2.265513e+03
std,2.937103e+04
min,1.000000e+00
25%,6.859925e+02
50%,1.172000e+03
75%,1.212385e+03
max,4.075401e+06


Exposure <= 0: 0
Exposure > 1: 1224
ClaimNb < 0: 0
ClaimAmount <= 0: 0


## 5. Reconcile policy claim counts with severity records

Count severity rows by `IDpol` and compare them with `ClaimNb`. This is a diagnostic comparison, not a join for modeling. Any differences require investigation before the files are combined.

In [6]:
severity_rows_per_policy = severity.groupby("IDpol").size().rename("SeverityRows")
claim_check = frequency[["IDpol", "ClaimNb"]].join(severity_rows_per_policy, on="IDpol")
claim_check["SeverityRows"] = claim_check["SeverityRows"].fillna(0).astype(int)
claim_check["Difference"] = claim_check["ClaimNb"] - claim_check["SeverityRows"]

print("Total reported claims:", frequency["ClaimNb"].sum())
print("Total severity records:", len(severity))
print("Policy rows with differing counts:", claim_check["Difference"].ne(0).sum())
display(claim_check.loc[claim_check["Difference"].ne(0)].head(10))

Total reported claims: 26444
Total severity records: 26444
Policy rows with differing counts: 0


,IDpol,ClaimNb,SeverityRows,Difference


## Findings and decisions

- **Dimensions and schema:** The CASdatasets exports contain 677,991 policy rows across 12 columns and 26,444 claim rows across 2 columns. No missing values were found in either table.
- **Policy IDs:** All frequency IDs are unique. One frequency ID is stored in scientific notation, causing pandas to infer a floating-point ID column by default. The notebook loads IDs as strings to preserve their stored values. All severity IDs match a frequency ID.
- **Exposure and outcomes:** Exposure is positive for every policy; 1,224 policies have exposure greater than one year, with a maximum of 2.01 years. Claim counts are nonnegative and claim amounts are positive. Claim amounts are strongly right-skewed, with a maximum of approximately 4.08 million. These observations warrant interpretation during analysis but do not, by themselves, justify removing records.
- **Claim reconciliation:** The sum of policy `ClaimNb` values equals the 26,444 severity records. Claim counts agree with the number of severity rows for every policy; there are no unmatched severity rows.
- **Decision:** Retain the current records for exploratory analysis. Continue to read `IDpol` as a string when linking the files, account for `Exposure` when calculating claim frequency, and assess the influence of large claims during severity analysis. These checks cleared the supplied records for the full-portfolio descriptive analyses in Notebooks 02 and 03; the policy-level development/test split was introduced later in Notebook 04.